# Baselines
- Naive Baseline
- Rolling Mean Baseline

## Data Loading and Preprocessing

In [9]:
# Load data and convert date string to datetime data
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("../data/portfolio_data.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df = df.set_index("Date")

df_returns = df.copy()
stock_cols = ['AMZN', 'DPZ', 'BTC', 'NFLX']

# Compute daily returns
df_returns[stock_cols] = df_returns[stock_cols].pct_change()
df_returns = df_returns.dropna()
df_returns.head()


,AMZN,DPZ,BTC,NFLX
Date,,,,
2013-05-02,0.017403,0.015556,-0.076706,0.007421
2013-05-03,0.021778,0.008830,0.150867,-0.004849
2013-05-06,-0.009029,0.014469,-0.029229,-0.012930
2013-05-07,0.007860,0.017785,0.032847,-0.021074
2013-05-08,0.003686,0.004325,-0.003534,0.011442


In [10]:
# Scale asset values between -1 and 1
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_returns[stock_cols])

In [11]:
# Split time series data into training samples
import numpy as np

def create_sequences(data, window_size=30):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, window_size=90)

## Naive Baseline predictions
We predict $r_{t+1}​=0$ where $r_t$ is the returns on a stock at time t. I.e., we predict tomorrow's returns will be the same as today's. 

In [ ]:
from sklearn.metrics import mean_squared_error


y_pred_naive = X[:, -1, :] # Last day in each window
mse_naive = mean_squared_error(y, y_pred_naive)
print(f'Naive MSE: {mse_naive}')

2.082946219556081

## Rolling Mean Baseline
Predict $r_{t+1} = 1/30 \sum_{i=t-29}^{t} r_i$

In [15]:
y_pred_rolling = X.mean(axis=1)
mse_rolling = mean_squared_error(y, y_pred_rolling)
print(f'Rolling Mean MSE: {mse_rolling}')

Rolling Mean MSE: 1.0366687827648025
